# Preprocessing and Harmonization

Clean, harmonize, and merge raw datasets into an analytical dataset.

## Setup

In [2]:
import sys
import pathlib
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

project_root = pathlib.Path().resolve().parent
sys.path.insert(0, str(project_root))

from src.preprocessing import load_ef_epi, clean_eurostat, compute_percentiles, merge_datasets, load_eurostat_data, export_merged

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## Load Raw Data

In [3]:
eurostat_raw = load_eurostat_data('educ_uoe_lang01', flags=False)
ef_raw = pd.read_csv(project_root / 'data' / 'raw' / 'efiepi_rankings.csv')

print(f'Eurostat: {eurostat_raw.shape}')
print(f'EF: {ef_raw.shape}')

Eurostat: (12133, 18)
EF: (246, 4)


## Clean Eurostat Data

Filter for English language, total sex, percentage units, and national-level (2-letter country codes).

In [4]:
eurostat_df = clean_eurostat(eurostat_raw)
print(f'Cleaned: {eurostat_df.shape}')
print(f'Columns: {list(eurostat_df.columns)}')
print(f'Countries: {eurostat_df["iso3"].nunique()}')
print(f'Year range: {int(eurostat_df["year"].min())} to {int(eurostat_df["year"].max())}')
print()
eurostat_df.head()

Cleaned: (365, 4)
Columns: ['geo', 'iso3', 'year', 'learning']
Countries: 34
Year range: 2012 to 2024



,geo,iso3,year,learning
0,AL,ALB,2024,96.433333
1,AT,AUT,2012,98.600000
2,AT,AUT,2013,99.540000
3,AT,AUT,2014,99.260000
4,AT,AUT,2015,99.480000


## Load and Clean EF Data

Harmonize country names to ISO 3166-1 alpha-3 codes.

In [5]:
ef_df = load_ef_epi(str(project_root / 'data' / 'raw' / 'efiepi_rankings.csv'))
print(f'Cleaned: {ef_df.shape}')
print(f'Columns: {list(ef_df.columns)}')
print(f'Countries: {ef_df["iso3"].nunique()}')
print(f'Year range: {int(ef_df["year"].min())} to {int(ef_df["year"].max())}')
print()
ef_df.head()

Country not found: Turkey
Country not found: Turkey
Country not found: Turkey
Country not found: Turkey
Country not found: Turkey
Country not found: Turkey
Country not found: Turkey
Country not found: Turkey
Cleaned: (238, 7)
Columns: ['year', 'Rank', 'country', 'score', 'country_clean', 'iso2', 'iso3']
Countries: 36
Year range: 2017 to 2024



,year,Rank,country,score,country_clean,iso2,iso3
0,2017,1,Netherlands,71.45,Netherlands,NL,NLD
1,2017,12,Belgium,61.58,Belgium,BE,BEL
2,2017,23,Greece,57.14,Greece,GR,GRC
3,2017,2,Sweden,70.40,Sweden,SE,SWE
4,2017,14,Switzerland,60.95,Switzerland,CH,CHE


## Compute Percentiles

Within each year, compute percentile ranks of learning exposure and EF proficiency to enable cross-year comparison.

In [6]:
eurostat_df = compute_percentiles(eurostat_df, 'learning', ['year'], 'learning_percentile')
ef_df = compute_percentiles(ef_df, 'score', ['year'], 'ef_percentile')

print('Percentiles computed')
print()
print('Eurostat percentile range:', eurostat_df['learning_percentile'].min(), 'to', eurostat_df['learning_percentile'].max())
print('EF percentile range:', ef_df['ef_percentile'].min(), 'to', ef_df['ef_percentile'].max())

Percentiles computed

Eurostat percentile range: 0.03125 to 1.0
EF percentile range: 0.03225806451612903 to 1.0


## Merge Datasets with Lag Alignment

Align EF proficiency (year t+4) with ESL exposure (year t) to reflect policy timing and skill development.

In [7]:
merged = merge_datasets(eurostat_df, ef_df, lag_years=4)

print(f'Merged shape: {merged.shape}')
print(f'Columns: {list(merged.columns)}')
print(f'Countries: {merged["iso3"].nunique()}')
print(f'Year range: {int(merged["year"].min())} to {int(merged["year"].max())}')
print()
merged.head(10)

Merged shape: (365, 7)
Columns: ['geo', 'iso3', 'year', 'learning', 'learning_percentile', 'ef_percentile', 'gap_pct']
Countries: 34
Year range: 2012 to 2024



,geo,iso3,year,learning,learning_percentile,ef_percentile,gap_pct
0,AL,ALB,2024,96.433333,0.645161,NaN,NaN
1,AT,AUT,2012,98.600000,1.000000,NaN,NaN
2,AT,AUT,2013,99.540000,0.900000,0.730769,-0.169231
3,AT,AUT,2014,99.260000,0.896552,0.709677,-0.186874
4,AT,AUT,2015,99.480000,0.931034,0.866667,-0.064368
5,AT,AUT,2016,99.680000,0.896552,NaN,NaN
6,AT,AUT,2017,99.540000,0.900000,0.967742,0.067742
7,AT,AUT,2018,99.420000,0.862069,0.966667,0.104598
8,AT,AUT,2019,99.360000,0.896552,0.967742,0.071190
9,AT,AUT,2020,99.540000,0.900000,NaN,NaN


## Gap Variable

gap_pct = EF_percentile − Learning_percentile  
Positive gap: high proficiency relative to learning exposure. Negative gap: low proficiency relative to exposure.

In [8]:
print('Gap statistics:')
print(merged['gap_pct'].describe())
print()
print('Non-null gap values:', merged['gap_pct'].notna().sum())

Gap statistics:
count    165.000000
mean       0.158061
std        0.394143
min       -0.733333
25%       -0.100000
50%        0.085057
75%        0.528365
max        0.900000
Name: gap_pct, dtype: float64

Non-null gap values: 165


## Validation

In [9]:
print('Validation:')
print(f'  Duplicates (iso3-year): {merged.duplicated(subset=["iso3", "year"]).sum()}')
print(f'  Missing values: {merged.isnull().sum().sum()}')
print(f'  Complete cases: {merged.dropna().shape[0]} / {merged.shape[0]} ({merged.dropna().shape[0]/merged.shape[0]*100:.1f}%)')
print(f'  EF match rate: {merged["ef_percentile"].notna().sum() / merged.shape[0] * 100:.1f}%')

Validation:
  Duplicates (iso3-year): 0
  Missing values: 400
  Complete cases: 165 / 365 (45.2%)
  EF match rate: 45.2%


## Export Dataset

In [10]:
output_path = project_root / 'data' / 'processed' / 'merged_analytical.csv'
output_path = export_merged(merged, output_path=output_path, project_root=project_root)

merged_sorted = merged.sort_values(['year', 'iso3']).reset_index(drop=True)
print(f'Exported to: {output_path}')
print(f'  Rows: {len(merged_sorted)}')
print(f'  Columns: {len(merged_sorted.columns)}')
print(f'  Size: {output_path.stat().st_size / 1024:.1f} KB')

Exported to: C:\Users\henri_ugzoq54\OneDrive\Workspace\2025_2026\Trento M1 DS\Semester 2\Data Vis\Project\Project Visualisation EF\data\processed\merged_analytical.csv
  Rows: 365
  Columns: 7
  Size: 19.9 KB
